# Python ও গণিত রিফ্রেশার

এই notebook-এ কোর্স জুড়ে ব্যবহৃত linear algebra, calculus ও probability-এর
বিল্ডিং block-গুলোর হাতে-কলমে demo করা হয়েছে: dot product, matrix
multiplication, norm, numerical বনাম analytical derivative, gradient,
chain rule এবং softmax / cross-entropy।

চালানোর নিয়ম:
    cell-গুলো উপরে থেকে নিচে চালাও (Shift + Enter)।


In [ ]:
"""
Python ও গণিত রিফ্রেশার

এই কোর্স জুড়ে ব্যবহৃত linear algebra, calculus ও probability-এর বিল্ডিং
block-গুলোর হাতে-কলমে demo: dot product, matrix multiplication, norm,
numerical বনাম analytical derivative, gradient, chain rule এবং
softmax / cross-entropy।

চালানোর নিয়ম:
    এই notebook-এর cell-গুলো উপরে থেকে নিচে চালাও (Shift + Enter)।
"""

import numpy as np


## 1. Linear algebra (লিনিয়ার অ্যালজেব্রা)

dot product, matrix multiplication, transpose ও norm — neural network-এর
প্রতিটি হিসাবের ভিত্তি। নিচের cell-এ এসব NumPy দিয়ে দেখা হয়েছে, সাথে
ম্যানুয়াল হিসাবের সাথে ফলাফল মিলিয়ে নেওয়া হচ্ছে।


In [ ]:
# ---------------------------------------------------------------------------
# 1. Linear algebra
# ---------------------------------------------------------------------------

def linear_algebra_demo():
    print("=" * 70)
    print("1. LINEAR ALGEBRA")
    print("=" * 70)

    a = np.array([1.0, 2.0, 3.0])
    b = np.array([4.0, 5.0, 6.0])

    # Dot product: ম্যানুয়ালি sum-of-products বনাম NumPy
    manual_dot = sum(ai * bi for ai, bi in zip(a, b))
    numpy_dot = np.dot(a, b)
    print(f"a = {a}, b = {b}")
    print(f"dot product (manual)  = {manual_dot}")
    print(f"dot product (np.dot)  = {numpy_dot}")
    assert np.isclose(manual_dot, numpy_dot)

    # Matrix multiplication — shapes অবশ্যই মেলাতে হবে: (d x n) @ (n x m) -> (d x m)
    W = np.array([[1.0, 0.0, -1.0],
                  [0.5, 2.0, 1.0]])          # shape (2, 3)
    x = np.array([1.0, 2.0, 3.0])            # shape (3,)
    y = W @ x                                # shape (2,)
    print(f"\nW shape = {W.shape}, x shape = {x.shape}")
    print(f"y = W @ x = {y}  (shape {y.shape})")

    # Transpose
    print(f"\nW^T =\n{W.T}")

    # Norms
    l2 = np.linalg.norm(a, ord=2)
    l1 = np.linalg.norm(a, ord=1)
    print(f"\n||a||_2 (Euclidean norm) = {l2:.4f}")
    print(f"||a||_1 (Manhattan norm) = {l1:.4f}")


linear_algebra_demo()


## 2. Calculus (ক্যালকুলাস): derivative, gradient, chain rule

numerical derivative-কে analytical (হাতে বের করা) derivative-এর সাথে
তুলনা করা হয়েছে; তারপর একটি multivariable function-এর gradient এবং
একটি composed function-এর উপরে chain rule যাচাই করা হয়েছে।


In [ ]:
# ---------------------------------------------------------------------------
# 2. Calculus: derivatives, gradients, chain rule
# ---------------------------------------------------------------------------

def numerical_derivative(f, x, h=1e-6):
    """f'(x)-এর central-difference আনুমানিক মান।"""
    return (f(x + h) - f(x - h)) / (2 * h)


def calculus_demo():
    print("\n" + "=" * 70)
    print("2. CALCULUS")
    print("=" * 70)

    # f(x) = x^2  ->  f'(x) = 2x
    f = lambda x: x ** 2
    x0 = 3.0
    analytic = 2 * x0
    numeric = numerical_derivative(f, x0)
    print(f"f(x) = x^2 at x=3 -> f'(x) analytic = {analytic}, numeric = {numeric:.6f}")

    # multivariable function-এর gradient: g(x, y) = x^2 + 3xy
    def g(x, y):
        return x ** 2 + 3 * x * y

    def grad_g(x, y, h=1e-6):
        dgdx = (g(x + h, y) - g(x - h, y)) / (2 * h)
        dgdy = (g(x, y + h) - g(x, y - h)) / (2 * h)
        return np.array([dgdx, dgdy])

    x0, y0 = 2.0, 1.0
    numeric_grad = grad_g(x0, y0)
    analytic_grad = np.array([2 * x0 + 3 * y0, 3 * x0])  # [dg/dx, dg/dy]
    print(f"grad(g)(2, 1) analytic = {analytic_grad}, numeric = {numeric_grad}")

    # Chain rule: y = f(u), u = g(x)  ->  dy/dx = f'(u) * g'(x)
    # ধরি g(x) = x^2 (u = x^2), f(u) = sin(u)  ->  y = sin(x^2)
    def compose(x):
        return np.sin(x ** 2)

    x0 = 1.5
    chain_rule_analytic = np.cos(x0 ** 2) * (2 * x0)   # f'(u)*g'(x)
    chain_rule_numeric = numerical_derivative(compose, x0)
    print(f"d/dx sin(x^2) at x=1.5 -> chain rule = {chain_rule_analytic:.6f}, "
          f"numeric = {chain_rule_numeric:.6f}")
    print("(This exact mechanism, applied automatically layer by layer, is backpropagation.)")


calculus_demo()


## 3. Probability (সম্ভাবনা): softmax এবং cross-entropy

README-র softmax ও log-likelihood সূত্রগুলো এখানে NumPy দিয়ে
বাস্তবায়িত হয়েছে — একটি model-এর মতো 4টি candidate next token-এর
logits-কে distribution-এ রূপান্তর করে cross-entropy loss বের করা হয়।


In [ ]:
# ---------------------------------------------------------------------------
# 3. Probability: softmax and cross-entropy
# ---------------------------------------------------------------------------

def softmax(logits):
    # numerical stability-র জন্য max বিয়োগ করা হলো (ফলাফল পরিবর্তন হয় না)।
    shifted = logits - np.max(logits)
    exps = np.exp(shifted)
    return exps / np.sum(exps)


def cross_entropy(probs, target_index):
    """`probs`-এর অধীনে সঠিক class-এর negative log-likelihood।"""
    return -np.log(probs[target_index] + 1e-12)


def probability_demo():
    print("\n" + "=" * 70)
    print("3. PROBABILITY")
    print("=" * 70)

    # ধরা যাক 4টি candidate next token-এর জন্য model এই logits ("scores")
    # তৈরি করেছে, softmax এগুলোকে distribution-এ রূপান্তর করার আগে।
    logits = np.array([2.0, 1.0, 0.1, -1.0])
    probs = softmax(logits)
    print(f"logits = {logits}")
    print(f"softmax(logits) = {np.round(probs, 4)}")
    print(f"sum(probs) = {probs.sum():.6f}  (always ~1.0 for a valid distribution)")

    true_next_token = 0  # ধরা যাক সঠিক next token হলো index 0
    loss = cross_entropy(probs, true_next_token)
    print(f"\nTrue next token index = {true_next_token}")
    print(f"Cross-entropy loss = -log(p_correct) = {loss:.4f}")
    print("This exact loss (cross-entropy over the vocabulary) is what every "
          "LLM in this course minimizes during pretraining and fine-tuning.")


probability_demo()


## সব একসাথে (main)

নিচের cell-এ পুরো স্ক্রিপ্টটি একবারে চালানো হয় — যেমনটি `python
example.py` চালালে হয়। (উপরের section cell-গুলোতে অংশগুলো আলাদাভাবে
দেখানো হয়েছে।)


In [ ]:
def main():
    linear_algebra_demo()
    calculus_demo()
    probability_demo()


main()
